# Spark SQL and DataFrames TP

## Imports and Initialization

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("Spark SQL TP") \
    .getOrCreate()

sc = spark.sparkContext

## Creating Dataframes

### Common ways
    
    1. From an RDD
    2. From a data source (csv, json, etc)
    3. From a table
    4. From an SQL statement

#### From an RDD

In [ ]:
first_rdd = sc.parallelize((
  (1, "Batman"),
  (2, "Superman"),
  (3, "Spiderman")
))

first_df = spark.createDataFrame(first_rdd)

#### Inspecting the schema

In [ ]:
# let's see how the dataframe and its schema look like

# first_df.show(5)
# first_df.schema  # schema object
first_df.printSchema()  # prints schema of dataframe

#### Creating a DataFrame from a data source / file

In [ ]:
df = (spark.read
      .option("sep", "\t")  # separator
      .option("header", False)  #  file has no header row
      .option("inferSchema", True)  # spark tries to infer data types
      .csv("../data/movie_data.tsv")
)

df.printSchema()

#### Creating/Changing a header
    First column - User ID
    Second column - movieID
    Third column - User rating (between 1-5)
    Fourth column - Timestamp

In [ ]:
movies_df = df.toDF("userID", "movieID", "rating", "timestamp")
movies_df.printSchema()

In [ ]:
movies_df.explain("simple")
# movies_df.explain("extended")
# movies_df.explain("formatted")

In [ ]:
movies_df.select("movieID", "rating").take(5)

In [ ]:
# movie IDs with rating >= 4
movies_df.select(movies_df.movieID).where("rating >= 4").distinct().count()

In [ ]:
from pyspark.sql.functions import col

movies_df.select(col("movieID")).take(5)

## Writing DataFrames

    1. Into tables/views
    2. Into files

### Storing DataFrames in Tables
The table will be accessible from other clusters and it will be persisted.

In [ ]:
(movies_df.write
 .mode("overwrite")  # ignore, append
 .saveAsTable("movie_rating")
)

spark.sql("SELECT COUNT(*) FROM movie_rating").show()

It is also possible to create a temporary, session-based view.

In [ ]:
movies_df.createOrReplaceTempView("temp_movie_rating")

spark.sql("SELECT COUNT(*) FROM temp_movie_rating").show()

In [ ]:
# read back a whole table into a DataFrame
movie_tables_df = spark.table("movie_rating")

display(movie_tables_df)

### Creating DataFrames with SQL

In [ ]:
movie_sql_df = spark.sql("SELECT movieID, rating, timestamp as cdate FROM movie_rating LIMIT 50")

display(movie_sql_df)
movie_sql_df.count()

### Using Parquet and Modes

In [ ]:
(first_df.write
  .option("compression", "snappy")
  .mode("overwrite")  # overwrites data if exists. other options "append", "error", "ignore"
  .parquet("/tmp/dataframe-training/first.parquet") # path
)

# test after running overwrite, append, ignore modes:
display(spark.read.parquet("/tmp/dataframe-training/first.parquet"))
spark.read.parquet("/tmp/dataframe-training/first.parquet").count()

## DataFrame Methods

### Column Transformations

#### Creating column objects

In [ ]:
from pyspark.sql.functions import col

movies_df.select(col("userID")).take(5)
# userID = movies_df.userID
# userID = movies_df["userID"]
# movies_df.select(userID).take(5)

#### Selecting columns

In [ ]:
userID_df = movies_df.select("userID", "rating")
# userID_df = movies_df.select(col("userID"), col("rating"))
# userID_df = movies_df.select(col("userID"), col("rating"))
# userID_df = movies_df.select(col("*"))  # selecting all values - SQL syntax

# Fields can be nested
# userID_df = movies_df.select(col("userID"), col("movieID"), col("timestamp.hour"))  # selecting nested fields (struct) with dot notation
# if the time stamp was a nested object with format hour, minute and seconds

display(userID_df)
userID_df.take(2)

#### Adding and Replacing Columns

In [ ]:
high_rating_df = movies_df.withColumn("high_rating", col("rating").isin(4, 5))

display(high_rating_df)
high_rating_df.take(15)

#### Renaming Columns

In [ ]:
time_df = movies_df.withColumnRenamed("timestamp", "time")
display(time_df)
display(movies_df)

#### Replacing Column Values

In [ ]:
from pyspark.sql.functions import when

high_rating = movies_df.select(
    col("*"),
    when(col("rating") == 5, 10)
    .when(col("rating") == 4, 8)
    .when(col("rating") == 3, 6)
    .alias("inflated_rating")
)
display(high_rating)
high_rating.show()

### Row Transformations
#### Filtering DataFrames

In [ ]:
low_rating = movies_df.filter("rating > 4")  # with RDDs: filter(lambda row: row["rating"] > 4)

low_rating.take(5)

In [ ]:
low_rating.count()

In [ ]:
movies_df.count()

#### Deduplicating Rows

In [ ]:
distinct_rows_df = movies_df.distinct()
distinct_rows_df.count()

# no_duplicates = spark.sql("SELECT DISTINCT(movieID) FROM movie_rating")
# no_duplicates.count()

In [ ]:
distinct_movies = movies_df.select("movieID").distinct()
distinct_movies.count()

#### Limiting

In [ ]:
limit_df = movies_df.limit(25)
display(limit_df)
limit_df.count()

#### Sorting

In [ ]:
increasing_ts_df = movies_df.sort("timestamp")
increasing_ts_df.take(5)

decreasing_ts_df = movies_df.sort(col("timestamp").desc())
decreasing_ts_df.take(5)
# display(decreasing_ts_df)

In [ ]:
# orderBy is same as sort

ordered_df = movies_df.orderBy(["movieID", "rating"])  # same as .sort(["movieID", "rating"])
ordered_df.take(5)

### Aggregations

In [ ]:
# creating grouped objects

movies_grp = movies_df.groupBy("movieID")
# user_grp = movies_df.groupBy("userID")

#### Counting

In [ ]:
# group and count

movies_grp.count().show()
# user_grp.count().show()

#### Average

In [ ]:
movies_df.groupBy("movieID").avg("rating").show()

#### Sum

In [ ]:
movies_df.groupBy("movieID").sum("rating").show()

#### More Complex Aggregations
- Use `agg()` for applying different types of aggregations
- Also allows for other transformations on top of the returned column

In [ ]:
from pyspark.sql.functions import sum, avg, approx_count_distinct

movies_df.groupBy("movieID").agg(avg("rating").alias("total_purchases")).show()

### Datetime Functions

#### Casting to DateTime

In [ ]:
from pyspark.sql.functions import to_date, from_unixtime, to_timestamp

timestamp_df = movies_df.withColumn("datetime", to_timestamp(from_unixtime(col("timestamp"))))

display(timestamp_df)
timestamp_df.show(5)

#### Formatting Dates

In [ ]:
from pyspark.sql.functions import date_format

format_df = (timestamp_df.withColumn("date string", date_format("datetime", "MMMM dd, yyyy"))
  .withColumn("time string", date_format("datetime", "HH:mm:ss.SSSSSS"))
) 
display(format_df)
format_df.show(5)

In [ ]:
format_df.explain("extended")

#### Extracting date parts

In [ ]:
from pyspark.sql.functions import year, month, dayofweek, minute, second

dt_col = col("datetime")

datetime_df = (timestamp_df
    .withColumn("year", year(dt_col))
    .withColumn("month", month(dt_col))
    .withColumn("dayofweek", dayofweek(dt_col))
    .withColumn("minute", minute(dt_col))
    .withColumn("second", second(dt_col))              
)

display(datetime_df)
datetime_df.show(5)

#### Converting to date

In [ ]:
from pyspark.sql.functions import to_date

date_df = timestamp_df.withColumn("date", to_date(col("datetime")))
display(date_df)
date_df.show(5)

#### Manipulating dates

In [ ]:
from pyspark.sql.functions import date_add

plus_df = timestamp_df.withColumn("plus_two_days", date_add(col("datetime"), 2))

# plus_df = timestamp_df.selectExpr("*","datetime + interval 2 days")  # spark sql allows for +/- interval type of datetime manipulation

display(plus_df)
plus_df.show(5)

## User Defined Functions (UDFs)

### Importing a New Dataset

In [ ]:
countries_df = spark.read.option("header", True).option("inferSchema", True).csv("../data/geo.csv")
# countries_df = spark.read.csv("../data/geo.csv", header=True, inferSchema=True)

display(countries_df)
countries_df.show(5)

### Calculating the Distance between two Latitude/Longitude Pairs
Source: https://gist.github.com/rochacbruno/2883505

In [ ]:
import math

from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

def distance(startLat, startLon, endLat, endLon):    

    radius = 6371  # in km
    dlat = math.radians(endLat - startLat)
    dlon = math.radians(endLon - startLon)
    a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(startLat)) \
        * math.cos(math.radians(endLat)) * math.sin(dlon/2) * math.sin(dlon/2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    d = radius * c

    return d

geo_distance = udf(distance, DoubleType()) # you can define the return type, default is string
geo_distance

In [ ]:
# Let's compare distances of each country's geographical centre, using our UDF

distance_df = (countries_df
  # .na.drop()
  # .filter(col("country") == "EE")
 .join(countries_df  # no join keys = cartesian/cross join
       # .filter(col("country") == "FI")
       .toDF("join_country","join_latitude","join_longitude","join_name")  # we can use .toDF() to rename all columns of a dataframe
       # .na.drop() # remove countries with null values
      )
 .withColumn("distance_in_km", geo_distance("latitude","longitude","join_latitude","join_longitude"))  # here we call the UDF
  
 # .filter(col("distance_in_km").cast(DoubleType()) > 0)
 # .orderBy("distance_in_km")
)

display(distance_df)
distance_df.show()

## Complex Types
 
We will use **another dataset** containing information about songs on Spotify.

For more data and a more detailed hands on, see https://github.com/anindya-saha/Data-Science-with-Spark/blob/master/working-with-nested-data-types/working-with-nested-data-types.ipynb

In [ ]:
spotify_df = spark.read.csv(path='../data/spotify-songs.csv', inferSchema=True, header=True)

In [ ]:
spotify_df.printSchema()

In [ ]:
spotify_df.show(10)

### Nested Types

Our goal is to

1. Combine the columns `'key', 'mode', 'target'` into an **array**.
   - An array contains a list of elements of _the same type_.
2. Transform the acoustic qualities
   ```
   'acousticness', 'tempo', 'liveness', 'instrumentalness', 'energy', 'danceability', 'speechiness', 'loudness'
   ```
    of a song from individual columns into a **map**.
    - A map contains key-value types.
    - All keys have to be of the same type, as do values.

In general, the keys for a map can be obtained from another column, but here we use the `lit` function to define constants (also called literals) instead.


In [ ]:
from pyspark.sql.functions import array, create_map, lit

spotify_map_df = (spotify_df.select(
    'id',
    'song_title',
    'artist',
    'duration_ms',
    array('key', 'mode', 'target').alias('audience'),
    create_map(
        lit('acousticness'), 'acousticness', 
        lit('danceability'), 'acousticness',
        lit('energy'), 'energy',
        lit('instrumentalness'), 'instrumentalness',
        lit('liveness'), 'liveness',
        lit('loudness'), 'loudness',
        lit('speechiness'), 'speechiness',
        lit('tempo'), 'tempo'
    ).alias('qualities'),
    'time_signature',
    'valence',
)  # .cache()   # Example use of cache
)

In [ ]:
spotify_map_df.printSchema()

In [ ]:
spotify_map_df.show(10, truncate=True)    # try with truncate=True (default value) or without truncate option

#### Write the new DataFrame to a JSON file

In [ ]:
spotify_map_df.coalesce(1).write.json(path='spotify-songs-json', mode="overwrite")

The `coalesce(1)` function reduces the number of partitions of the DataFrame to 1.

Since Spark creates one file for each partition, by using `coalesce(1)`, you're asking Spark to write entire DataFrame into a single JSON file.

Note: Using `coalesce(1)` is typically done when you want a single output file, which is common in cases where you need a compact, single-file format for easier consumption or when a system expects a single file as output. However, this can hurt performance when working with large datasets, as it forces Spark to consolidate everything into a single partition.

#### Reload the restructured DataFrame using a more complex schema with Nested Data Types

In [ ]:
from pyspark.sql.types import StructType, StructField, ArrayType, MapType, StringType, IntegerType

nested_schema = StructType((
    StructField('id', IntegerType(), nullable=False),
    StructField('song_title', StringType(), nullable=False),
    StructField('artist', StringType(), nullable=False),
    StructField('duration_ms', IntegerType(), nullable=False),
    StructField('audience',
                ArrayType(elementType=IntegerType()),
                nullable=False
    ),
    StructField('qualities',
                MapType(keyType=StringType(), valueType=DoubleType(), valueContainsNull=False),
                nullable=True
    ),
    StructField('time_signature', IntegerType(), nullable=False),
    StructField('valence', DoubleType(), nullable=False),
))

In [ ]:
spotify_reloaded_df = spark.read.json(path='spotify-songs-json', schema=nested_schema)

In [ ]:
spotify_reloaded_df.show(10, truncate=False)

In [ ]:
spotify_reloaded_df.printSchema()

#### Extracting array elements
Recall that `audience` column is a combination of three attributes `key`, `mode`, and `target`.
We can extract each element into its own column again.

In [ ]:
spotify_map_df.select(
    'song_title',
    col('audience').getItem(0).alias('key'), 
    col('audience').getItem(1).alias('mode'),
    col('audience').getItem(2).alias('target'),
).show(5)

#### Extracting map values
Recall that the `qualities` column is a map created from multiple attributes of a song.
We can extract (some of) these attributes again and store them in individual columns.

In [ ]:
spotify_map_df.select(
    'song_title',
    col('qualities').getItem('acousticness').alias('acousticness'),
    col('qualities').getItem('speechiness').alias('speechiness'),
).limit(10).show()

#### A bit more condensed

In [ ]:
# For a more concise and to generate a more efficient parsed logical plan.

cols = [col("song_title")] + list(map(
        lambda f: col("qualities").getItem(f).alias(str(f)), ["acousticness", "speechiness", "liveness", "tempo"]))

spotify_map_df.select(cols).show(10)

#### Extracting map values programmatically
In five steps.

Manually appending the columns is fine if we know all the distinct keys in the map.
If we don’t know all the distinct keys, we’ll need a programatic solution, but be warned – this approach is slow!

**Step 1: Create a DataFrame with all the unique keys**

In [ ]:
from pyspark.sql.functions import collect_set, map_keys, explode

keys_df = spotify_map_df.select(explode(map_keys(col("qualities")))).distinct()

In [ ]:
keys_df.show(truncate=False)

In [ ]:
keys_df.explain()

**Step 2: Convert the DataFrame to a list with all the unique keys**

In [ ]:
keys = [row[0] for row in keys_df.collect()]
keys

**Step 3: Create an array of column objects for the map items**

In [ ]:
key_cols = list(map(lambda f: col("qualities").getItem(f).alias(str(f)), keys))

**Step 4: Add any additional columns before calculating the final result**

In [ ]:
final_cols = [col("song_title")] + key_cols

**Step 5: Run a `select()` to get the final result**

In [ ]:
spotify_map_df.select(final_cols).show(10)

In [ ]:
spotify_map_df.select(final_cols).explain(True)

## The End

In [ ]:
spark.stop()